# Hourly Hospital Demand Forecasting

## 1. Objective

The objective of this model is to forecast hourly hospital visit demand over a seven-day horizon using historical visit records.

A Random Forest Regressor is trained on historical hourly demand, calendar features, and lagged demand variables to identify temporal patterns and estimate future patient arrivals.

The resulting forecasts are intended to support hospital resource planning through a capacity simulation dashboard.

**Forecasting approach:**
- **Target variable:** Hourly visit demand (`Demand`).
- **Model:** Random Forest Regressor.
- **Forecast horizon:** 168 hours (7 days).
- **Data source:** `VISIT_OCCURRENCE`.
- **Evaluation:** Chronological holdout, MAE, RMSE, R², and seasonal naive baseline.

**Scope:** The model forecasts visit demand, not patient waiting times, treatment durations, or actual hospital staffing requirements. Capacity planning is performed separately using hypothetical operational assumptions.

In [ ]:
import pandas as pd
import sklearn

## 2. Data Preparation

### 2.1 Data Source

The forecasting model uses the VISIT_OCCURRENCE table, which contains historical hospital visit records.

The original dataset includes 1,706,001 visits, with timestamps ranging from January 1, 2023, to August 13, 2026.

The visit_start_datetime column was selected to represent the time at which each hospital visit was recorded.

In [ ]:
visit_occurrence = pd.read_csv("../../data/visit_occurrence_202611061448.csv")

### 2.2 Hourly Demand Aggregation

Individual visit records were aggregated into hourly intervals using Pandas resample("h").

Each resulting observation represents the total number of hospital visits recorded during a specific hour.

The resulting time series includes hours with zero recorded visits, preserving the hourly frequency required for forecasting.

In [ ]:
visit_occurrence["visit_start_datetime"] = pd.to_datetime(visit_occurrence["visit_start_datetime"])
df_sorted = visit_occurrence.sort_values("visit_start_datetime")
df_hours = (df_sorted.set_index("visit_start_datetime").resample("h").size().reset_index(name="Demand"))

df_hours = df_hours[df_hours["visit_start_datetime"] <= "2026-07-16 23:59:59"]
demand = df_hours["Demand"].sum()
zero_demand_rows = df_hours[df_hours["Demand"] == 0]

print(df_hours.head(5),df_hours.tail(5))
print("Total rows:", df_hours.shape)
print(f"Null values:\n{df_hours.isnull().sum()}")
print(f"""Total records in visit_occurrence: {len(visit_occurrence["visit_occurrence_id"])} 
Total records in demand: {demand}
""")
print("\nFirst hours with zero demand:")
print(len(zero_demand_rows))

### 2.3 Temporal Data Quality

A significant change in data coverage was identified after July 17, 2026, at 18:00.

Before this point, records showed consistent hourly activity. Afterward, observations became sporadic, with substantial temporal gaps and repeated demand values of two visits.

This pattern suggests a potential interruption or change in the data collection or extraction process.

To avoid introducing potentially unreliable demand patterns, the modeling dataset was restricted to January 1, 2023, through July 16, 2026, at 23:00.

After applying this restriction, 1,704,561 visits were retained, representing approximately 99.92% of the original records.

### 2.4 Missing Values

The hourly demand series was checked for missing values.

Zero-demand hours were preserved because they may represent legitimate periods of low hospital activity.

Rows without sufficient historical information for the lagged and rolling features were excluded later during feature engineering.

## 3. Feature Engineering

Feature engineering was performed to capture temporal patterns and historical demand behavior. Seven predictive features were created from the hourly demand time series.

### 3.1 Calendar Features

Four calendar-based features were extracted from `visit_start_datetime`:

| Feature | Description |
|---|---|
| `hour` | Hour of the day (0–23). |
| `day_of_week` | Day of the week (0 = Monday, 6 = Sunday). |
| `month` | Month of the year (1–12). |
| `is_weekend` | Binary indicator identifying Saturdays and Sundays. |

These features allow the model to identify hourly, weekly, and seasonal demand patterns.

### 3.2 Historical Demand Features

Three additional features were created using historical hourly demand:

| Feature | Description |
|---|---|
| `lag_168` | Number of visits recorded at the same hour one week earlier. |
| `lag_336` | Number of visits recorded at the same hour two weeks earlier. |
| `rolling_mean_168` | Average hourly demand over a 168-hour historical window, shifted by 168 hours. |

The historical features were designed to support a seven-day forecasting horizon without requiring actual demand observations from the forecast period.

### 3.3 Handling Missing Values

Lagged and rolling features introduce missing values at the beginning of the time series because sufficient historical observations are not available.

Rows containing these initial missing values were removed using `dropna()`.

The resulting modeling dataset contains **30,696 hourly observations and nine columns**, with no remaining missing values.

In [ ]:
df_hours["hour"] = df_hours["visit_start_datetime"].dt.hour

df_hours["day_of_week"] = df_hours["visit_start_datetime"].dt.dayofweek

df_hours["month"] = df_hours["visit_start_datetime"].dt.month

df_hours["is_weekend"] = (
    df_hours["visit_start_datetime"].dt.dayofweek >= 5
).astype(int)

df_hours["lag_168"] = df_hours["Demand"].shift(168)
df_hours["lag_336"] = df_hours["Demand"].shift(336)

df_hours["rolling_mean_168"] = df_hours["Demand"].shift(168).rolling(168).mean()

df_model = df_hours.dropna()
print(df_model.shape)
print(df_model.isnull().sum())


## 4. Model Training and Evaluation

### 4.1 Chronological Train-Test Split

The modeling dataset was divided chronologically to preserve the temporal order of observations and prevent training on future target values.

The final **672 hours (four weeks)** were reserved for testing, while the preceding **30,024 hourly observations** were used for training.

This evaluation strategy allows the model to be assessed on later observations, simulating four consecutive seven-day forecasting periods.

Each evaluation week represents a seven-day forecasting horizon. Historical demand features are calculated using observed demand available before the start of each weekly period. Therefore, the evaluation simulates weekly forecast updates rather than a single uninterrupted four-week forecast.

In [ ]:
#Data division

df_train = df_model.iloc[:-672]
df_test = df_model.iloc[-672:]

features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_168",
    "lag_336",
    "rolling_mean_168"
]

X_train = df_train[features]
y_train = df_train["Demand"]
X_test = df_test[features]
y_test = df_test["Demand"]

print("X_train.shape: ",X_train.shape)
print("y_train.shape: ",y_train.shape)
print("X_test.shape: ",X_test.shape)
print("y_test.shape: ",y_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 4.2 Random Forest Regressor

A Random Forest Regressor was selected to model nonlinear relationships between calendar features, historical demand, and hourly hospital visits.

The model was configured with the following hyperparameters:

| Parameter | Value |
|---|---|
| `n_estimators` | 100 |
| `max_depth` | 15 |
| `random_state` | 42 |
| `n_jobs` | -1 |

The model was trained using seven predictive features, with hourly visit demand (`Demand`) as the target variable.


In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


### 4.3 Model Performance

Model performance was evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the coefficient of determination (R²).

| Metric | Random Forest |
|---|---:|
| MAE | 13.57 visits/hour |
| RMSE | 27.69 visits/hour |
| R² | 0.7708 |

The model explained approximately **77.08% of the variability** in hourly demand during the test period.

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

### 4.4 Baseline Comparison

A seasonal naive baseline was implemented using the observed demand from the same hour of the previous week (`lag_168`).

| Metric | Baseline | Random Forest |
|---|---:|---:|
| MAE | 18.09 | 13.57 |
| RMSE | 36.68 | 27.69|
| R² | 0.5980 | 0.7708|

Random Forest reduced MAE by approximately **25%** compared with the baseline.

In [ ]:
baseline_pred = X_test["lag_168"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = mean_squared_error(y_test, baseline_pred) ** 0.5
baseline_r2 = r2_score(y_test, baseline_pred)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

### 4.5 Weekly Evaluation

The test period was divided into four consecutive blocks of 168 hours to evaluate performance across individual weekly forecasting horizons.

| Week | RF MAE | Baseline MAE | RF RMSE | RF R² |
|---|---:|---:|---:|---:|
| 1 | 12.676 | 18.310 | 22.111 | 0.861 |
| 2 | 15.659 | 18.155 | 33.181 | 0.597 |
| 3 | 10.339 | 17.417 | 17.126 | 0.921 |
| 4 | 15.601 | 18.488 | 34.416 | 0.650 |

Random Forest achieved a lower MAE than the seasonal naive baseline in all four evaluation weeks.

The strongest performance was observed in Week 3 (R² = 0.921), while Week 2 showed the lowest coefficient of determination (R² = 0.597).

These results demonstrate that the model captures useful temporal patterns, although its predictive performance varies across different periods.

In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

for week in range(4):
    start = week * 168
    end = start + 168

    # Datos reales y predicciones de la semana
    y_real_week = y_test.iloc[start:end]
    y_pred_week = y_pred[start:end]
    baseline_week = baseline_pred.iloc[start:end]

    # Métricas semanales
    results.append({
        "Week": week + 1,
        "RF_MAE": mean_absolute_error(y_real_week, y_pred_week),
        "Baseline_MAE": mean_absolute_error(y_real_week, baseline_week),
        "RF_RMSE": mean_squared_error(y_real_week, y_pred_week) ** 0.5,
        "RF_R2": r2_score(y_real_week, y_pred_week)
    })

df_results = pd.DataFrame(results)

print(df_results.round(3))

## 5. Final Model and Weekly Forecast

### 5.1 Final Model Training

After evaluating the model on the chronological test set, the Random Forest Regressor was retrained using all available modeling observations.

This approach allows the final model to learn from the complete historical dataset before generating future predictions.

The final model retains the same hyperparameters used during evaluation.

The previously reported test metrics are preserved because evaluating the retrained model on the same observations would produce in-sample results rather than an independent performance estimate.

In [ ]:
features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_168",
    "lag_336",
    "rolling_mean_168"
]

X_model = df_model[features]
y_model = df_model["Demand"]

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_model, y_model)


### 5.2 Seven-Day Forecast Generation

A seven-day forecast was generated for the period from **July 17, 2026, at 00:00 to July 23, 2026, at 23:00**.

The forecasting dataset contains 168 consecutive hourly timestamps.

Calendar features were calculated from the future timestamps, while historical demand features were constructed using observations available before the forecast period.

The trained model then generated one predicted demand value for each hour.

This forecast is retrospective and represents the seven days following the last retained historical observation, rather than the current calendar week.

In [ ]:
future_dates = pd.date_range(
    start=df_hours["visit_start_datetime"].max() + pd.Timedelta(hours=1),
    periods=168,
    freq="h"
)

df_future = pd.DataFrame({
    "visit_start_datetime": future_dates
})

df_future["hour"] = df_future["visit_start_datetime"].dt.hour

df_future["month"] = df_future["visit_start_datetime"].dt.month

df_future["day_of_week"] = df_future["visit_start_datetime"].dt.day_of_week

df_future["is_weekend"] = (
    df_future["visit_start_datetime"].dt.day_of_week >= 5
).astype(int)

historical_demand = df_hours["Demand"].to_numpy()

df_future["lag_168"] = historical_demand[-168:]
df_future["lag_336"] = historical_demand[-336:-168]

df_future["rolling_mean_168"] = [
    historical_demand[i:i + 168].mean()
    for i in range(len(historical_demand) - 335, len(historical_demand) - 167)
]

X_future = df_future[features]

future_predictions = model.predict(X_future)

df_future["Predicted_Demand"] = future_predictions

print(df_future[["visit_start_datetime", "Predicted_Demand"]].head(10))
print(df_future.isnull().sum())
print("Total hours:", len(df_future))



### 5.3 Forecast Export

The final predictions were exported to `hourly_demand_forecast.csv` for integration into the Power BI dashboard.

The exported dataset contains 168 hourly observations and two columns:

| Column | Description |
|---|---|
| `visit_start_datetime` | Date and hour associated with the forecast. |
| `Predicted_Demand` | Estimated number of hospital visits during that hour. |

Predicted demand values were rounded to non-negative integers to represent visit counts.

The CSV is excluded from version control and shared separately with the dashboard development team, subject to the project's data-sharing requirements.

In [ ]:
df_export = df_future[
    ["visit_start_datetime", "Predicted_Demand"]
].copy()

df_export["Predicted_Demand"] = (
    df_export["Predicted_Demand"]
    .clip(lower=0)
    .round()
    .astype(int)
)

df_export.to_csv(
    "../../reports/hourly_demand_forecast.csv",
    index=False
)

print(df_export.head())
print("Total rows:", len(df_export))

In [ ]:
print("Minimum predicted demand:", df_future["Predicted_Demand"].min())
print("Maximum predicted demand:", df_future["Predicted_Demand"].max())
print("Average predicted demand:", df_future["Predicted_Demand"].mean())
print("Negative predictions:", (df_future["Predicted_Demand"] < 0).sum())

## 6. Limitations and Conclusions

### 6.1 Data Limitations

The forecasting model has several limitations related to the available hospital data:

- **Temporal data quality:** A significant reduction in data coverage was detected after July 17, 2026. Therefore, the modeling period was restricted to observations ending on July 16, 2026.
- **Limited operational information:** The dataset does not provide reliable information about actual staffing levels, service capacity, or patient waiting times.
- **Unmodeled external factors:** Holidays, unexpected events, and other external factors that may influence hospital demand were not explicitly included as predictive features.
- **Forecast uncertainty:** The model produces point estimates rather than prediction intervals, so uncertainty is not explicitly quantified.
- **Historical forecasting period:** The generated forecast covers July 17–23, 2026, and should not be interpreted as a prediction of current hospital demand.

These limitations should be considered when interpreting the forecasts and using them for operational planning.

### 6.2 Model Conclusions

The Random Forest Regressor demonstrated promising performance in forecasting hourly hospital visit demand using calendar features and historical demand patterns.

The model achieved the following results on the chronological test set:

| Metric | Random Forest | Seasonal Naive Baseline |
|---|---:|---:|
| MAE | 13.57 | 18.09 |
| RMSE | 27.69 | 36.68 |
| R² | 0.7708 | 0.5980 |

The model explained approximately **77.08% of the variability** in hourly hospital demand during the test period.

Compared with the seasonal naive baseline, Random Forest reduced the Mean Absolute Error by approximately **25%**, demonstrating an improvement over predictions based solely on demand from the previous week.

Weekly evaluation showed that Random Forest achieved a lower MAE than the baseline in all four evaluation periods. However, performance varied across weeks, with R² values ranging from 0.597 to 0.921.

After evaluation, the model was retrained using all available modeling observations to generate a seven-day forecast.

The final output consists of **168 hourly demand predictions**, covering July 17–23, 2026. These predictions were exported to a CSV file for integration into the Power BI dashboard.

Overall, the results suggest that machine learning can provide useful short-term demand estimates to support hospital resource planning. However, the predictions should be used as decision-support inputs rather than as direct measurements of staffing requirements or operational efficiency.

### 6.3 Future Improvements

Several improvements could be explored in future iterations:

- Incorporating additional operational variables, such as staffing levels, service capacity, and patient waiting times, when reliable data becomes available.
- Including holidays and other external factors that may influence hospital demand.
- Evaluating alternative forecasting algorithms and performing time-series-aware hyperparameter optimization.
- Implementing prediction intervals to quantify forecast uncertainty.
- Applying rolling-origin backtesting across additional historical periods to assess forecasting stability.
- Retraining and monitoring the model as new, reliable hospital visit records become available.

These improvements could increase forecasting reliability and strengthen the connection between predicted demand and hospital resource allocation decisions.